# Accessibility Audit Pipeline — Step-by-Step Walkthrough

This notebook walks through every stage of the element-specific accessibility audit pipeline,
using the trimmed Vision Aid homepage (`dat_visionaid_home.html`, ~143 KB) as our test file.

The pipeline has five stages:

| Stage | What happens | Cost |
|-------|-------------|------|
| **Step 0** — Programmatic Checks | Rule-based detection of missing alt, empty links, duplicate IDs, etc. | Free |
| **Step 1** — Extract | Three extractors parse HTML into structured JSON payloads, discarding layout noise | Free |
| **Step 2** — Slice | Each payload is cut into targeted pieces — one per evaluation prompt | Free |
| **Step 3** — Fill Templates | Each slice is inserted into a focused prompt template | Free |
| **Step 4** — Call LLM | Each filled prompt is sent to the Anthropic API | ~$0.05–$0.32 total |

By the end, you will understand exactly what data goes into each LLM call and why.

---
## Setup

First we clone the repository and install its dependencies. If you're running this
locally from the project root, you can skip the clone cell — but on Google Colab
it's required.

In [ ]:
# Clone the repo and install dependencies (required on Google Colab)
# If running locally from the project root, skip this cell.

import os

if os.getenv("COLAB_RELEASE_TAG"):
    !git clone -b feat/llm-prompt-pipeline https://github.com/marimdz/visionaid-a11y-llm-audit.git
    %cd visionaid-a11y-llm-audit
    !pip install -e . -q
    print("\nSetup complete — repo cloned and dependencies installed.")
else:
    print("Not running on Colab — skipping clone. Make sure you're in the project root.")

In [ ]:
import sys
import json
from pathlib import Path
from collections import Counter

# Resolve project root (this notebook lives at the project root)
PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

HTML_FILE = PROJECT_ROOT / "test_files" / "dat_visionaid_home.html"

def estimate_tokens(text: str) -> int:
    """Rough token estimate: ~4 characters per token for English/HTML."""
    return max(1, len(text) // 4)

print(f"Project root : {PROJECT_ROOT}")
print(f"HTML file    : {HTML_FILE.name}  ({HTML_FILE.stat().st_size / 1024:.0f} KB)")
print(f"File exists  : {HTML_FILE.exists()}")

---
## How big is the raw HTML?

Before we extract anything, let's see how large this file is in terms of tokens.
LLMs have context windows (e.g., 200k tokens for Claude), and sending raw HTML is wasteful
because most of the file is CSS classes, `<div>` wrappers, and JavaScript — none of which
matter for accessibility evaluation.

In [ ]:
raw_text = HTML_FILE.read_text(encoding="utf-8", errors="replace")
raw_bytes = HTML_FILE.stat().st_size
raw_tokens = estimate_tokens(raw_text)

print(f"File size   : {raw_bytes:>10,} bytes  ({raw_bytes / 1024:.0f} KB)")
print(f"Characters  : {len(raw_text):>10,}")
print(f"Est. tokens : {raw_tokens:>10,}")
print()
print(f"At Claude Sonnet pricing ($3/million input tokens):")
print(f"  Sending this raw HTML once = ${raw_tokens * 3 / 1_000_000:.4f}")
print()
print("The extractors will reduce this dramatically by keeping only")
print("the semantically relevant elements (headings, links, forms, images, etc.).")

---
## Step 0 — Programmatic Checks

Before involving the LLM at all, we run rule-based checks that can definitively
detect certain accessibility failures. These are binary pass/fail — no judgment needed.

The programmatic checker (`processing_scripts/programmatic/semantic_checklist_01.py`)
looks for:

| Issue Code | WCAG | What It Detects |
|------------|------|-----------------|
| `MISSING_H1` | 1.3.1 | No `<h1>` tag on the page |
| `IMG_MISSING_ALT` | 1.1.1 | `<img>` with no `alt` attribute |
| `IMG_EMPTY_ALT` | 1.1.1 | `<img alt="">` (possibly mis-marked as decorative) |
| `LINK_NO_TEXT` | 2.4.4 | `<a>` with no text and no `aria-label` |
| `DUPLICATE_ID` | 4.1.1 | Multiple elements sharing the same ID |
| `IFRAME_NO_TITLE` | 4.1.2 | `<iframe>` missing or empty title |

In [ ]:
from processing_scripts.programmatic.semantic_checklist_01 import audit_html_file

programmatic_findings = audit_html_file(str(HTML_FILE))

print(f"Total programmatic findings: {len(programmatic_findings)}")
print()

# Count by issue type
issue_counts = Counter(f["rule_id"] for f in programmatic_findings)
print(f"{'Rule ID':<30} {'Count':>6}")
print(f"{'-'*40}")
for rule_id, count in issue_counts.most_common():
    print(f"{rule_id:<30} {count:>6}")

In [ ]:
# Let's look at a few example findings
print("Sample programmatic findings (first 5):\n")
for f in programmatic_findings[:5]:
    print(f"  [{f['rule_id']}] {f['rule_name']}")
    print(f"    {f['description']}")
    print()

These findings are free (no API cost) and definitive. They'll be combined with
LLM findings in the final report.

---
## Step 1 — Extract Structured Payloads

Now we run the three HTML extractors. Each one uses BeautifulSoup to parse the HTML
and extract only the elements relevant to its checklist, outputting a structured JSON dict.

| Extractor | Focus Area | Key Output Fields |
|-----------|-----------|------------------|
| **CL01** — Semantic Structure | `semantic_checklist_01.py` | `page_title`, `headings`, `flagged_links`, `landmarks`, `tables`, `iframes` |
| **CL02** — Forms | `forms_checklist_02.py` | `forms[].fields` (with `label_source`, `effective_label`, `instructions`, `required`), `forms[].groups` |
| **CL03** — Non-text Content | `nontext_checklist_03.py` | `images` (informative/decorative/actionable/complex), `svgs`, `icon_fonts`, `media` |

The extractors discard all CSS, JavaScript, and layout wrappers — keeping only what
the LLM needs to evaluate WCAG compliance.

In [ ]:
from processing_scripts.llm_preprocessing.semantic_checklist_01 import extract as cl01_extract
from processing_scripts.llm_preprocessing.forms_checklist_02 import extract as cl02_extract
from processing_scripts.llm_preprocessing.nontext_checklist_03 import extract as cl03_extract

cl01_payload = cl01_extract(str(HTML_FILE))
cl02_payload = cl02_extract(str(HTML_FILE))
cl03_payload = cl03_extract(str(HTML_FILE))

# Token sizes
cl01_json = json.dumps(cl01_payload)
cl02_json = json.dumps(cl02_payload)
cl03_json = json.dumps(cl03_payload)

cl01_tokens = estimate_tokens(cl01_json)
cl02_tokens = estimate_tokens(cl02_json)
cl03_tokens = estimate_tokens(cl03_json)
combined_tokens = cl01_tokens + cl02_tokens + cl03_tokens

print(f"{'Checklist':<25} {'~Tokens':>8}  {'% of raw':>8}")
print(f"{'-'*45}")
print(f"{'CL01 — Semantic':<25} {cl01_tokens:>8,}  {cl01_tokens/raw_tokens*100:>7.1f}%")
print(f"{'CL02 — Forms':<25} {cl02_tokens:>8,}  {cl02_tokens/raw_tokens*100:>7.1f}%")
print(f"{'CL03 — Non-text':<25} {cl03_tokens:>8,}  {cl03_tokens/raw_tokens*100:>7.1f}%")
print(f"{'-'*45}")
print(f"{'Combined':<25} {combined_tokens:>8,}  {combined_tokens/raw_tokens*100:>7.1f}%")
print(f"{'Raw HTML':<25} {raw_tokens:>8,}  {'100.0%':>8}")
print()
reduction = (1 - combined_tokens / raw_tokens) * 100
print(f"Token reduction: {reduction:.0f}%")

### CL01 — Semantic Structure Payload

Let's see what the CL01 extractor found. This covers headings, links, landmarks, tables, and iframes.

In [ ]:
print("CL01 Payload — Section summary")
print(f"{'Section':<22} {'Items':>6}")
print(f"{'-'*32}")
for key, val in cl01_payload.items():
    if isinstance(val, list):
        print(f"{key:<22} {len(val):>6}")
    elif isinstance(val, dict):
        if key == "images":
            total = sum(len(v) for v in val.values())
            print(f"{key:<22} {total:>6}  (subcategories: {list(val.keys())})")
        else:
            print(f"{key:<22} {'(dict)':>6}")
    else:
        print(f"{key:<22} {repr(val)[:30]:>30}")

### CL02 — Forms Payload

The forms extractor provides rich metadata per field: how the label is associated
(`label_source`), the text of the label (`effective_label`), whether it's required, etc.

In [ ]:
forms = cl02_payload["forms"]
all_fields = [f for form in forms for f in form["fields"]]
all_groups = [g for form in forms for g in form["groups"]]
required_fields = [f for f in all_fields if f.get("required")]

# Label source distribution
src_counts = Counter(f["label_source"] for f in all_fields)

print(f"Total forms    : {len(forms)}")
print(f"Total fields   : {len(all_fields)}")
print(f"Total groups   : {len(all_groups)}")
print(f"Required fields: {len(required_fields)}")
print()
print("Label source breakdown:")
for src, count in src_counts.most_common():
    flag = "  <-- problem" if src in ("placeholder_only", "none") else ""
    print(f"  {src:<22} {count:>4}{flag}")

### CL03 — Non-text Content Payload

Images are split into four categories because the evaluation criteria differ for each:

- **Informative**: Alt text must accurately describe the *content*
- **Decorative**: Empty alt is correct only if the image is truly decorative
- **Actionable**: Alt must describe the *action/destination*, not the image appearance
- **Complex**: Charts/diagrams need extended descriptions

In [ ]:
img = cl03_payload["images"]
total_images = sum(len(v) for v in img.values())

print(f"Images — informative : {len(img['informative'])}")
print(f"Images — decorative  : {len(img['decorative'])}")
print(f"Images — actionable  : {len(img['actionable'])}  (inside <a> or <button>)")
print(f"Images — complex     : {len(img['complex'])}")
print(f"Total images         : {total_images}")
print(f"SVGs (non-hidden)    : {len(cl03_payload['svgs'])}")
print(f"Icon fonts           : {len(cl03_payload['icon_fonts'])}")
print(f"Media elements       : {len(cl03_payload['media'])}")
print()

# Show a few examples
if img["informative"]:
    print("Sample informative images (first 3):")
    for im in img["informative"][:3]:
        print(f"  alt={repr(im['alt'][:60])}")

if img["actionable"]:
    print("\nSample actionable images (first 3):")
    for im in img["actionable"][:3]:
        lbl = im.get("link_aria_label") or im.get("link_text") or im.get("button_text") or "NONE"
        print(f"  [{im['context']}] alt={repr(str(im['alt'])[:35])}  parent_label={repr(lbl[:30])}")

---
## Step 2 — The Prompt Registry

The prompt registry (`processing_scripts/llm/registry.py`) is the single source of truth
for what the pipeline evaluates. It defines 21 `PromptSpec` entries, each mapping:

- **name** — unique identifier (e.g., `"link_clarity"`)
- **checklist** — which extractor payload to use (`"CL01"`, `"CL02"`, or `"CL03"`)
- **prompt_file** — path to the `.txt` file containing the prompt template
- **prompt_index** — which numbered prompt within that file (1-based)
- **payload_slicer** — name of the function that extracts the right JSON slice
- **wcag_criteria** — WCAG success criteria this prompt evaluates
- **skip_if_empty** — if the slice has no data, skip the API call entirely

In [ ]:
from processing_scripts.llm.registry import PROMPT_REGISTRY

print(f"Total prompts in registry: {len(PROMPT_REGISTRY)}")
print()
print(f"{'Name':<30} {'CL':>3}  {'WCAG':<20} {'Type':<8} {'Summary?'}")
print(f"{'-'*80}")
for spec in PROMPT_REGISTRY:
    wcag = ", ".join(spec.wcag_criteria)
    summary = "Yes" if spec.is_summary else ""
    print(f"{spec.name:<30} {spec.checklist:>3}  {wcag:<20} {spec.output_type:<8} {summary}")

---
## Step 3 — Slice Payloads

Each prompt needs only a *subset* of its checklist's full payload. The slicer functions
(`processing_scripts/llm/slicers.py`) extract exactly the data each prompt needs.

For example:
- The `page_title` prompt only needs the `page_title` object from CL01
- The `link_clarity` prompt only needs the `flagged_links` array
- The `decorative_verification` prompt only needs the `images.decorative` array from CL03

This slicing is what keeps each API call small and focused.

In [ ]:
from processing_scripts.llm.slicers import get_slicer, is_empty_slice

# Store all payloads keyed by checklist name, matching what run_pipeline.py does
payloads = {"CL01": cl01_payload, "CL02": cl02_payload, "CL03": cl03_payload}

# Slice every prompt and show token sizes
print(f"{'Prompt':<30} {'~Tokens':>8}  {'Empty?':>6}")
print(f"{'-'*50}")

total_slice_tokens = 0
non_empty_count = 0

for spec in PROMPT_REGISTRY:
    if spec.is_summary:
        continue  # skip summaries for this walkthrough
    slicer_fn = get_slicer(spec.payload_slicer)
    payload_json = slicer_fn(payloads[spec.checklist])
    tokens = estimate_tokens(payload_json)
    empty = is_empty_slice(payload_json)
    
    status = "SKIP" if empty else ""
    print(f"{spec.name:<30} {tokens:>8,}  {status:>6}")
    
    if not empty:
        total_slice_tokens += tokens
        non_empty_count += 1

print(f"{'-'*50}")
print(f"Non-empty prompts: {non_empty_count}")
print(f"Total payload tokens (non-empty): ~{total_slice_tokens:,}")

Let's look at one slice in detail to see what data the LLM actually receives.
We'll pick the `page_title` slice (small and easy to read):

In [ ]:
# Show the page_title slice — this is all the LLM sees for this prompt
page_title_slicer = get_slicer("slice_page_title")
page_title_json = page_title_slicer(payloads["CL01"])

print("Page title slice (what the LLM receives):")
print(page_title_json)

In [ ]:
# And a more interesting one — flagged links
link_slicer = get_slicer("slice_flagged_links")
links_json = link_slicer(payloads["CL01"])
links_data = json.loads(links_json)

print(f"Flagged links slice: {len(links_data)} links, ~{estimate_tokens(links_json):,} tokens")
print()
print("These links were flagged because their text is short (≤2 words),")
print("generic ('click here', 'read more'), or missing entirely.")
print("The LLM evaluates whether the link text is descriptive enough.\n")
print("First 5 flagged links:")
for link in links_data[:5]:
    aria = link.get('aria_label') or ''
    aria_str = f"  aria_label={repr(aria)}" if aria else ""
    print(f"  text={repr(link.get('text', '')):<30}{aria_str}")

---
## Step 4 — Fill Prompt Templates

The prompt templates live in `.txt` files under `processing_scripts/llm/`. Each file
contains multiple prompts separated by dashed headers:

```
  ---------------------------------
  1) PAGE TITLE EVALUATION PROMPT
  ---------------------------------

  You are an accessibility auditor.
  Evaluate the page title.
  ...
  Data: {payload}
```

The `templates.py` module parses these files, finds the right prompt by index,
and replaces `{payload}` with the sliced JSON from the previous step.

In [ ]:
from processing_scripts.llm.templates import fill_template, get_template

# Let's look at one template BEFORE filling it
page_title_spec = PROMPT_REGISTRY[0]  # page_title
raw_template = get_template(page_title_spec)

print(f"Prompt: {page_title_spec.name}")
print(f"File: {page_title_spec.prompt_file}")
print(f"Index: {page_title_spec.prompt_index}")
print(f"WCAG: {page_title_spec.wcag_criteria}")
print()
print("--- Raw template (before payload substitution) ---")
print(raw_template)

In [ ]:
# Now fill it with actual data
filled_prompt = fill_template(page_title_spec, page_title_json)

print("--- Filled prompt (what gets sent to the LLM) ---")
print(filled_prompt)
print()
print(f"Total tokens: ~{estimate_tokens(filled_prompt):,}")

### All prompts — filled and ready

Let's fill every non-empty, non-summary prompt and see the token cost of each.
This is exactly what `run_pipeline.py` does before sending to the API.

In [ ]:
filled_prompts = {}  # name -> (spec, filled_text)
skipped = []

for spec in PROMPT_REGISTRY:
    if spec.is_summary:
        skipped.append((spec.name, "summary (optional)"))
        continue
    
    slicer_fn = get_slicer(spec.payload_slicer)
    payload_json = slicer_fn(payloads[spec.checklist])
    
    if spec.skip_if_empty and is_empty_slice(payload_json):
        skipped.append((spec.name, "empty payload"))
        continue
    
    filled_text = fill_template(spec, payload_json)
    filled_prompts[spec.name] = (spec, filled_text)

# Display results
total_tokens = 0
print(f"{'Prompt':<30} {'~Tokens':>8}  {'WCAG':<20}")
print(f"{'-'*62}")
for name, (spec, text) in filled_prompts.items():
    tokens = estimate_tokens(text)
    total_tokens += tokens
    wcag = ", ".join(spec.wcag_criteria)
    print(f"{name:<30} {tokens:>8,}  {wcag:<20}")

print(f"{'-'*62}")
print(f"{'TOTAL':<30} {total_tokens:>8,}")
print(f"{'API calls needed':<30} {len(filled_prompts):>8}")
print()

# Cost estimates
sonnet_cost = total_tokens * 3 / 1_000_000
print(f"Estimated input cost (Sonnet @ $3/MTok): ${sonnet_cost:.4f}")
print()

if skipped:
    print(f"Skipped ({len(skipped)}):")
    for name, reason in skipped:
        print(f"  {name}: {reason}")

---
## Step 5 — Preview a Prompt (Dry Run)

Let's look at one more filled prompt in full — the `heading_structure` prompt —
to see how the extracted data gets woven into the evaluation instructions.

In a live run, this exact text would be sent to the Anthropic API.
In dry-run mode, it gets saved to `output/prompts/heading_structure.json` for inspection.

In [ ]:
if "heading_structure" in filled_prompts:
    spec, text = filled_prompts["heading_structure"]
    print(f"Prompt: {spec.name}")
    print(f"WCAG: {spec.wcag_criteria}")
    print(f"Tokens: ~{estimate_tokens(text):,}")
    print()
    # Show first 2000 chars to avoid flooding the notebook
    preview = text[:2000]
    print(preview)
    if len(text) > 2000:
        print(f"\n... ({len(text) - 2000} more characters) ...")
else:
    print("heading_structure prompt was skipped (empty payload).")

---
## Summary — Token Budget

Here's a complete picture of the token reduction achieved by extraction + slicing.

In [ ]:
print("=" * 56)
print("  PIPELINE TOKEN SUMMARY")
print("=" * 56)
print(f"  Raw HTML file              : ~{raw_tokens:>8,} tokens")
print(f"  After extraction (3 JSONs) : ~{combined_tokens:>8,} tokens  ({combined_tokens/raw_tokens*100:.1f}% of raw)")
print(f"  After slicing (prompts)    : ~{total_tokens:>8,} tokens  ({total_tokens/raw_tokens*100:.1f}% of raw)")
print()
print(f"  Prompts to send            : {len(filled_prompts):>8}")
print(f"  Prompts skipped            : {len(skipped):>8}")
print(f"  Programmatic findings      : {len(programmatic_findings):>8}  (free)")
print()
print(f"  Estimated cost (Sonnet)    : ${total_tokens * 3 / 1_000_000:.4f} input")
print(f"  vs. sending raw HTML       : ${raw_tokens * 3 / 1_000_000:.4f} input")
print(f"  Savings                    : {(1 - total_tokens/raw_tokens) * 100:.0f}%")

---
## Automated Orchestration: `run_pipeline.py`

Everything we did above step-by-step is automated by `entry_points/run_pipeline.py`.
It runs the full pipeline with a single command:

### Dry run (no API calls, no cost)

Generates all prompts and saves them as JSON files for inspection:

```bash
python entry_points/run_pipeline.py --html test_files/dat_visionaid_home.html --dry-run
```

### Live run (calls the Anthropic API)

Sends each prompt to the LLM and saves the responses:

```bash
python entry_points/run_pipeline.py --html test_files/dat_visionaid_home.html
```

### Generate a CSV report

After a live run, combine programmatic + LLM findings into a single CSV:

```bash
python entry_points/generate_report.py
```

### What `run_pipeline.py` does

The script performs the exact same steps we walked through above:

1. **Step 0** — Runs `analyze_html()` for programmatic checks (free)
2. **Step 1** — Calls all three extractors to produce structured JSON payloads
3. **Step 2** — For each of the 21 prompts in the registry:
   - Slices the relevant payload subset
   - Skips if the slice is empty (e.g., no tables on the page = skip table prompt)
   - Fills the prompt template with the sliced data
   - Calls the Anthropic API (or saves the prompt in dry-run mode)
4. **Step 3** — Writes a `manifest.json` with run metadata, token counts, and cost estimates

### Pipeline options

| Flag | Default | Description |
|------|---------|-------------|
| `--html` | (required) | Path to the HTML file to analyze |
| `--output-dir` | `./output` | Directory for results |
| `--model` | `claude-sonnet-4-20250514` | Anthropic model to use |
| `--dry-run` | off | Generate prompts without calling the API |
| `--include-summaries` | off | Include the 3 cross-cutting summary prompts |
| `--show-cost` | off | Print estimated dollar cost after the run |

### Output structure

```
output/
├── manifest.json                 # Run metadata, token counts, prompt status
├── programmatic_findings.json    # Rule-based checker results (free)
├── payloads/                     # Raw extractor output (for debugging)
│   ├── cl01_payload.json
│   ├── cl02_payload.json
│   └── cl03_payload.json
└── prompts/                      # One file per prompt
    ├── page_title.json           # Contains prompt text, payload slice, and API response
    ├── heading_structure.json
    └── ...
```